In [2]:
import csv
import os

In [8]:
with open(os.path.normpath(os.path.join(os.getcwd(), "../", "../", "csv", "cloud_inventory.csv")), 'r') as f:
    # reader = csv.reader(f) 
    reader = csv.reader(f)
    print(reader)
    # reader.
    # next(reader)  # Skip the header row

    for vm_id, region, status, uptime_days, monthly_cost in reader:
        # print(row)


        total_cost = (float(uptime_days)//12)*12
        if (region == "us-east-1" and float(uptime_days)>=30):
            print(vm_id, region, status, uptime_days, monthly_cost,total_cost)
            

i-2eef4462 us-east-1 pending 347 271.57 336.0
i-43d38b9f us-east-1 terminated 151 172.55 144.0
i-3987e4ca us-east-1 terminated 280 297.92 276.0
i-d3ca7766 us-east-1 stopped 144 156.83 144.0
i-c8c4d1a6 us-east-1 stopped 231 212.3 228.0
i-59711ad1 us-east-1 running 125 304.99 120.0
i-185649e2 us-east-1 stopped 175 55.19 168.0
i-e044014f us-east-1 stopped 184 17.57 180.0
i-2b36059f us-east-1 stopped 357 110.93 348.0
i-739e63f1 us-east-1 stopped 248 421.38 240.0
i-c14f3b2d us-east-1 stopped 261 154.98 252.0
i-b9c4d737 us-east-1 running 258 453.74 252.0
i-572ce3eb us-east-1 pending 176 223.92 168.0
i-bb17dc8e us-east-1 pending 151 194.42 144.0
i-547ad9c6 us-east-1 running 196 248.53 192.0
i-c10968f6 us-east-1 pending 49 153.98 48.0
i-a1c93fe5 us-east-1 pending 179 467.35 168.0
i-7acd7ce0 us-east-1 running 355 318.01 348.0
i-d05f5f64 us-east-1 running 240 139.95 240.0
i-c687a62e us-east-1 running 297 227.72 288.0
i-d7a0724a us-east-1 pending 316 370.6 312.0
i-8b1707ef us-east-1 stopped 191 3

In [22]:
with open('output.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    # writer.writerow(['ID', 'Status'])  # Write a single row
    
    for row in reader:
        writer.writerows(row) 

ValueError: I/O operation on closed file.

In [ ]:
import csv
import os

# 1. Properly locate the path
file_path = os.path.normpath(os.path.join(
    os.getcwd(), "../../csv/cloud_inventory.csv"))
data_to_write = []

# 2. Read and store data
with open(file_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["vm_id"])
        # Store the data so we can use it outside this block
        data_to_write.append(row)

# 3. Write data to new file
with open('output.csv', 'w', newline='') as f:
    # If using standard writer, you define columns manually
    writer = csv.writer(f)
    writer.writerow(['ID', 'Status'])  # Header

    for row in data_to_write:
        # row is a dict, so we grab specific keys or all values
        writer.writerow([row["vm_id"], "Processed"])

In [26]:
fields = ['id', 'region']
data = [{'id': 'vm-123', 'region': 'us-east'},
        {'id': 'vm-123', 'region': 'us-east'}]

with open('export.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(data)

In [ ]:
import csv
from collections import defaultdict,Counter


# def clean_cpu(val):
#     try:
#         # Strip whitespace and check if it's a "null-like" string
#         if str(val).strip().upper() == "NULL" or val is None:
#             return 0.0
#         return float(val)
#     except (ValueError, TypeError):
#         return 0.0
    
region_based_dict = defaultdict(set)
cluster_based_dict = defaultdict(set)

cluster_error_rate = Counter()
cluster_correct_rate = Counter()
with open("/Users/aryansingh/Documents/devops/csv/telemetry.csv","r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        vm_id = row["vm_id"]
        cluster_id = row["cluster_id"]
        region = row["region"]
        cpu_usage = row["cpu_usage"]
        monthly_cost = float(row.get("monthly_cost") or 0)
        ncpu_usage = 0.0
        if cpu_usage == "NULL" or cpu_usage == "":
            cluster_error_rate[cluster_id] += 1
            ncpu_usage = float(0.0)
        else:
            cluster_correct_rate[cluster_id] += 1
            ncpu_usage = float(cpu_usage)
        
        cluster_based_dict[cluster_id].add((vm_id,ncpu_usage,monthly_cost))
        region_based_dict[region].add(cluster_id)
        
    #need to find a cluster inside which all vms have usage less than 10%
    # cluster_based_dict.items()


valid_clusters = set()



for cluster_id in cluster_based_dict.keys():
    cluster_flag = True
    for vm_id, cpu_usage, monthly_cost in cluster_based_dict[cluster_id]:
        if cpu_usage < 50:
            pass
        else:
            cluster_flag = False
            break
    if cluster_flag:
        valid_clusters.add(cluster_id)
        

result = set()
for valid_cluster_id in valid_clusters:
    counter = 0
    if(((cluster_error_rate[valid_cluster_id]/cluster_correct_rate[valid_cluster_id]) or 1)*100 < 5):
    # we need to check if it spans in more than 1 region
        for region, clusters_list in region_based_dict.items():
            if(valid_cluster_id in clusters_list):
                counter+=1
            if counter >= 2:
                result.add(valid_cluster_id)
                break
    else:
        # invalid cluster
        pass
    
# check for cluster:
# print(cluster_error_rate,cluster_correct_rate)
    
# print(cluster_based_dict)
# print(cluster_based_dict.items())
# print(region_based_dict)
# print(region_based_dict.items())
    
print(result)    


# Writing a simple summary
with open("decommission_report.txt", "w") as f:
    f.write(f"Clusters identified: {len(result)}\n")
    for cluster in result:
        f.write(f"ID: {cluster}\n")

{'cluster-002'}


In [41]:
with open("decommission_report.txt", "r") as f:
    # reader = f.readline()
    for line in f:
        print(line)

Clusters identified: 1

ID: cluster-002



In [39]:
0 or 1

1